In [1]:
import os
import re
import pandas as pd

# ==================== Configuration Area ====================
# Paths for SetA
setA_list_path = "instanceSetA.txt"         # List file of SetA instance names
folder_path_A = "SetA"      # Folder where SetA result files are located
output_csv_A = "SetA_results_summary.csv"   # Output path for SetA statistical results

# Paths for SetB
setB_list_path = "instanceSetB.txt"         # List file of SetB instance names
folder_path_B = "SetB"      # Folder where SetB result files are located
output_csv_B = "SetB_results_summary.csv"   # Output path for SetB statistical results
# ============================================================

def calculate_statistics(data):
    best_obj = min(o for o, _ in data)
    avg_obj = sum(o for o, _ in data) / len(data)
    avg_time = sum(t for _, t in data) / len(data)
    return best_obj, avg_obj, avg_time

def process_and_save_set(list_file_path, folder_path, prefix, output_csv_path):
    """
    Process a single set of data and save it as an independent CSV file.
    """
    if not os.path.exists(list_file_path):
        print(f"Warning: Instance list file '{list_file_path}' not found. Skipping this set.")
        return

    if not os.path.exists(folder_path):
        print(f"Warning: Result folder '{folder_path}' not found. Skipping this set.")
        return

    # Read instance names
    with open(list_file_path, "r", encoding="utf-8") as f:
        instances = [line.strip() for line in f if line.strip()]

    results = []

    # Process instances sequentially
    for inst_name in instances:
        # Filename remains in "SetA_instanceName.txt" or "SetB_instanceName.txt" format
        file_name = f"{prefix}_{inst_name}.txt"
        file_path = os.path.join(folder_path, file_name)
        
        if not os.path.exists(file_path):
            print(f"Warning: File '{file_path}' not found. Skipped.")
            continue

        results_list = []
        current_obj = None

        # Read and parse the file
        with open(file_path, "r", encoding="utf-8") as f_in:
            for line in f_in:
                line = line.strip()

                mO = re.search(r"O:\s*(\d+)", line)
                if mO:
                    current_obj = int(mO.group(1))

                mT = re.search(r"T:\s*([\d\.]+)s", line)
                if mT and current_obj is not None:
                    current_time = float(mT.group(1))
                    results_list.append((current_obj, current_time))
                    current_obj = None

        # Calculate statistics
        if len(results_list) > 0:
            best_obj, avg_obj, avg_time = calculate_statistics(results_list)
            results.append({
                "instance": f"{prefix}_{inst_name}",  # Keep prefix in the exported instance name
                "Best": best_obj,
                "Avg": avg_obj,
                "AvgT": avg_time
            })
        else:
            print(f"Info: No valid data extracted from file '{file_name}'.")

    # Save as an independent CSV
    if results:
        results_df = pd.DataFrame(results)
        results_df.to_csv(output_csv_path, index=False)
        print(f"[{prefix}] Statistical results saved to: {output_csv_path}")
    else:
        print(f"[{prefix}] No valid data collected. CSV was not generated.")


# Execute processing
print("Starting to process SetA...")
process_and_save_set(setA_list_path, folder_path_A, "SetA", output_csv_A)

print("\nStarting to process SetB...")
process_and_save_set(setB_list_path, folder_path_B, "SetB", output_csv_B)

Starting to process SetA...

Starting to process SetB...
